<img src="images/images_0.png" width="800" height="400">

### Step 1: Install Required Libraries


In [1]:
!pip install langchain-classic langchain-community  transformers sentence-transformers faiss-cpu pypdf -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 101.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 87.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 6.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


### Step 2: Import required Libraries + Config

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModelForQuestionAnswering
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline
from langchain_classic.chains import RetrievalQA, ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferMemory
from langchain_community.llms import HuggingFaceHub
import torch
import os
from google.colab import userdata

hf_token = userdata.get("HF_TOKEN")

### Step 3: Load the PDF document and split it into chunks

<img src="images/images_1.png" width="400" height="150">

<img src="images/images_2.png" width="400" height="300">

<img src="images/images_7.png" width="650" height="700">

In [3]:
loader = PyPDFLoader("/content/HR Policy Manual 2025.pdf")
text_splitter = RecursiveCharacterTextSplitter(
                                          chunk_size=1000,
                                          chunk_overlap=50,
                                          separators=["\n\n\n", "\n\n", ";","."," "]
                                              )


docs = loader.load_and_split(text_splitter=text_splitter)

### Step 4: Create embeddings and store them in a vector database

<img src="images/images_3.png" width="800" height='400'>

In [ ]:
emb_kwargs={'model_name' :"sentence-transformers/all-MiniLM-l6-v2",
'model_kwargs':{'device':'cuda'},
'encode_kwargs':{'normalize_embeddings': False}}

embeddings = HuggingFaceEmbeddings(**emb_kwargs)

<img src="images/images_4.png" width="600" height='300'>

In [5]:
#vector store
db = FAISS.from_documents(docs,embeddings)

$$cos(\theta)=\frac{A.B}{||A|| \ ||B||}=\frac{\sum_{i=1}^n  A_i B_i}{\sqrt{\sum_{i=1}^n  A_i^2} \sqrt{\sum_{i=1}^n B_i^2}}$$

<img src="images/images_5.png" width="400" height="150">

In [6]:
question = "How many types of leave are available?"
searchDocs = db.similarity_search(question)
print(searchDocs[0].page_content)

..............................................................................56
5.2 Leave Type 2: Earned Leave  ............................................................................57
5.3 Leave Type 3: Half Pay Leave ........................................................................... 58
5.4 Leave Type 4: Commuted Leave  ......................................................................58
5.5 Leave Type 5: Extraordinary Leave ....................................................................59
5.6 Leave Type 6: Maternity Leave  ......................................................................... 60
5.7 Leave Type 7: Paternity Leave .......................................................................... 60
5.8 Leave Type 8: Leave to female employees on adoption of child ...........................61
(6) Encashment of Earned Leave ....................................................................61
(7) Public Holidays and Restricted Holidays ...............

<img src="images/images_6.png" width="600" height='300'>

### Step 5: Load the LLM and create the RetrievalQA chain

In [ ]:
device = torch.device('cuda')

checkpoint = "MBZUAI/LaMini-T5-738M"
print(f"Checkpoint path: {checkpoint}")  # Add this line for debugging
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    checkpoint,
    device_map=device,
    torch_dtype=torch.float32
)
pipe = pipeline(
        "text-generation",
        model = base_model,
        tokenizer = tokenizer,
        max_length = 256,
        do_sample = True,
        temperature = 0.3,
        top_p= 0.95,
    )

llm = HuggingFacePipeline(pipeline=pipe)


In [8]:

#llm = HuggingFaceHub(repo_id="MBZUAI/LaMini-T5-738M", model_kwargs={"temperature":0.5, "max_length": 512})
memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)
conversation_chain = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=db.as_retriever(),
        memory=memory
    )

/tmp/ipykernel_9287/2027552613.py:2: LangChainDeprecationWarning: Please see the migration guide at: https://python.langchain.com/docs/versions/migrating_memory/
  memory = ConversationBufferMemory(memory_key="chat_history", return_messages=True)


In [9]:
result = conversation_chain.invoke({
    "question": "What insurance benefits are covered?"
})
print(result["answer"])

Token indices sequence length is longer than the specified maximum sequence length for this model (853 > 512). Running this sequence through the model will result in indexing errors


Use the following pieces of context to answer the question at the end. If you don't know the answer, just say that you don't know, don't try to make up an answer.

. Below is the sum insured.
Sr. No. Category Sum Insured
01 Group A / Faculty members 30 Lacs
02 Group A / Manager 30 Lacs
03 Group B / Assistant Manager 25 Lacs
04 Group C / Executive 20 Lacs
05 Group D employees 15 Lacs
Besides the above sum insured, the policy will have coverage of weekly benefits, OPD/IPD expense 
in case of accident and children education benefit in case of death. These benefits will depend 
upon the benefits provided by the Insurance Company.

94
IIMA HR Policy Manual 2025
ADDITIONAL TOP-UP GROUP MEDICLAIM INSURANCE SCHEME (PARENT’S 
POLICY)
Institute has introduced the additional top-up on the base policy for parents, which any  faculty 
or staff member can opt for their parents. The individual can opt for following coverage:
Group Existing Coverage under 
GMIS (In Rs.)
Additional Top-up coverage (In 